In [1]:
# %pip install pandas numpy matplotlib seaborn scikit-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
import warnings

warnings.filterwarnings('ignore')

In [3]:
# Load the CSV files
teams = pd.read_csv('data/wteams.csv')
sub = pd.read_csv('final_submission_2025.csv')

# Display the first few rows of each dataframe
print(teams.head())
print(sub.head())

   TeamID     TeamName
0    3101  Abilene Chr
1    3102    Air Force
2    3103        Akron
3    3104      Alabama
4    3105  Alabama A&M
               ID      Pred
0  2025_1101_1102  0.797561
1  2025_1101_1103  0.075156
2  2025_1101_1104  0.177275
3  2025_1101_1105  0.330844
4  2025_1101_1106  0.531553


In [4]:
# Split the 'ID' column into three separate columns
sub[['Year', 'Team1', 'Team2']] = sub['ID'].str.split('_', expand=True)

# Display the updated dataframe
print(sub.head())

               ID      Pred  Year Team1 Team2
0  2025_1101_1102  0.797561  2025  1101  1102
1  2025_1101_1103  0.075156  2025  1101  1103
2  2025_1101_1104  0.177275  2025  1101  1104
3  2025_1101_1105  0.330844  2025  1101  1105
4  2025_1101_1106  0.531553  2025  1101  1106


In [5]:
# Convert Team1 and Team2 columns to integers
sub['Team1'] = sub['Team1'].astype(int)
sub['Team2'] = sub['Team2'].astype(int)

# Merge sub with teams to get Team1Name
sub = sub.merge(teams[['TeamID', 'TeamName']], left_on='Team1', right_on='TeamID', how='left').rename(columns={'TeamName': 'Team1Name'})

# Merge sub with teams to get Team2Name
sub = sub.merge(teams[['TeamID', 'TeamName']], left_on='Team2', right_on='TeamID', how='left').rename(columns={'TeamName': 'Team2Name'})

# Drop the TeamID columns as they are no longer needed
sub = sub.drop(columns=['TeamID_x', 'TeamID_y'])

# Display the updated dataframe
print(sub.head())

               ID      Pred  Year  Team1  Team2 Team1Name Team2Name
0  2025_1101_1102  0.797561  2025   1101   1102       NaN       NaN
1  2025_1101_1103  0.075156  2025   1101   1103       NaN       NaN
2  2025_1101_1104  0.177275  2025   1101   1104       NaN       NaN
3  2025_1101_1105  0.330844  2025   1101   1105       NaN       NaN
4  2025_1101_1106  0.531553  2025   1101   1106       NaN       NaN


In [6]:
# Add a new column for the prediction if Team1 and Team2 were switched
sub['Pred_Switched'] = 1 - sub['Pred']

# Display the updated dataframe
print(sub.head())

               ID      Pred  Year  Team1  Team2 Team1Name Team2Name  \
0  2025_1101_1102  0.797561  2025   1101   1102       NaN       NaN   
1  2025_1101_1103  0.075156  2025   1101   1103       NaN       NaN   
2  2025_1101_1104  0.177275  2025   1101   1104       NaN       NaN   
3  2025_1101_1105  0.330844  2025   1101   1105       NaN       NaN   
4  2025_1101_1106  0.531553  2025   1101   1106       NaN       NaN   

   Pred_Switched  
0       0.202439  
1       0.924844  
2       0.822725  
3       0.669156  
4       0.468447  


In [7]:
result = sub[(sub['Team1'] == 1102) & (sub['Team2'] == 1101)]['Pred'].values
if result.size > 0:
	pred_value = result[0]
	print(pred_value)
else:
	print("No matching records found.")

No matching records found.


In [8]:
team1_rows = sub[sub['Team1'] == 1277]
print(team1_rows)

                   ID      Pred  Year  Team1  Team2 Team1Name Team2Name  \
46956  2025_1277_1278  0.810036  2025   1277   1278       NaN       NaN   
46957  2025_1277_1279  0.624171  2025   1277   1279       NaN       NaN   
46958  2025_1277_1280  0.551020  2025   1277   1280       NaN       NaN   
46959  2025_1277_1281  0.420814  2025   1277   1281       NaN       NaN   
46960  2025_1277_1282  0.830565  2025   1277   1282       NaN       NaN   
...               ...       ...   ...    ...    ...       ...       ...   
47146  2025_1277_1476  0.850180  2025   1277   1476       NaN       NaN   
47147  2025_1277_1477  0.926909  2025   1277   1477       NaN       NaN   
47148  2025_1277_1478  0.850472  2025   1277   1478       NaN       NaN   
47149  2025_1277_1479  0.891867  2025   1277   1479       NaN       NaN   
47150  2025_1277_1480  0.878745  2025   1277   1480       NaN       NaN   

       Pred_Switched  
46956       0.189964  
46957       0.375829  
46958       0.448980  
46959  

In [9]:
michigan_team = teams[teams['TeamName'].str.contains('Michigan', case=False, na=False)]
print(michigan_team)

     TeamID     TeamName
40     3141   C Michigan
83     3185   E Michigan
174    3276     Michigan
175    3277  Michigan St
341    3444   W Michigan


In [10]:
florida_duke_teams = teams[teams['TeamName'].isin(['Florida', 'Duke'])]
print(florida_duke_teams)

    TeamID TeamName
79    3181     Duke
94    3196  Florida


In [11]:
michigan_matches = sub[(sub['Team1Name'].isin(['Michigan', 'Michigan St'])) | (sub['Team2Name'].isin(['Michigan', 'Michigan St']))]
print(michigan_matches)

                    ID      Pred  Year  Team1  Team2    Team1Name  \
66231   2025_3101_3276  0.148800  2025   3101   3276  Abilene Chr   
66232   2025_3101_3277  0.087995  2025   3101   3277  Abilene Chr   
66591   2025_3102_3276  0.179567  2025   3102   3276    Air Force   
66592   2025_3102_3277  0.109607  2025   3102   3277    Air Force   
66950   2025_3103_3276  0.078968  2025   3103   3276        Akron   
...                ...       ...   ...    ...    ...          ...   
112681  2025_3277_3476  0.826903  2025   3277   3476  Michigan St   
112682  2025_3277_3477  0.942408  2025   3277   3477  Michigan St   
112683  2025_3277_3478  0.948194  2025   3277   3478  Michigan St   
112684  2025_3277_3479  0.962742  2025   3277   3479  Michigan St   
112685  2025_3277_3480  0.985386  2025   3277   3480  Michigan St   

             Team2Name  Pred_Switched  
66231         Michigan       0.851200  
66232      Michigan St       0.912005  
66591         Michigan       0.820433  
66592      

In [12]:
minnesota_rutgers_matches = sub[(sub['Team1Name'].isin(['Minnesota', 'Rutgers'])) | (sub['Team2Name'].isin(['Minnesota', 'Rutgers']))]
print(minnesota_rutgers_matches)

                    ID      Pred  Year  Team1  Team2    Team1Name  \
66233   2025_3101_3278  0.232706  2025   3101   3278  Abilene Chr   
66305   2025_3101_3353  0.537995  2025   3101   3353  Abilene Chr   
66593   2025_3102_3278  0.128182  2025   3102   3278    Air Force   
66665   2025_3102_3353  0.473053  2025   3102   3353    Air Force   
66952   2025_3103_3278  0.063358  2025   3103   3278        Akron   
...                ...       ...   ...    ...    ...          ...   
124142  2025_3353_3476  0.662636  2025   3353   3476      Rutgers   
124143  2025_3353_3477  0.725925  2025   3353   3477      Rutgers   
124144  2025_3353_3478  0.676760  2025   3353   3478      Rutgers   
124145  2025_3353_3479  0.602810  2025   3353   3479      Rutgers   
124146  2025_3353_3480  0.481947  2025   3353   3480      Rutgers   

             Team2Name  Pred_Switched  
66233        Minnesota       0.767294  
66305          Rutgers       0.462005  
66593        Minnesota       0.871818  
66665      

In [13]:
# Rename columns for clarity
df = sub.rename(columns={
    'ID': 'MatchID',
    'Pred': 'PredictionScore',
    'Year': 'MatchYear',
    'Team1': 'Team1ID',
    'Team2': 'Team2ID',
    'Team1Name': 'Team1Name',
    'Team2Name': 'Team2Name',
    'Pred_Switched': 'PredictionScoreSwitched'
})

# Create a new column to indicate the winning team based on the prediction score
df['WinningTeam'] = np.where(df['PredictionScore'] > 0.50, df['Team1Name'], df['Team2Name'])

# Display the updated dataframe
print(df.head())

          MatchID  PredictionScore MatchYear  Team1ID  Team2ID Team1Name  \
0  2025_1101_1102         0.797561      2025     1101     1102       NaN   
1  2025_1101_1103         0.075156      2025     1101     1103       NaN   
2  2025_1101_1104         0.177275      2025     1101     1104       NaN   
3  2025_1101_1105         0.330844      2025     1101     1105       NaN   
4  2025_1101_1106         0.531553      2025     1101     1106       NaN   

  Team2Name  PredictionScoreSwitched WinningTeam  
0       NaN                 0.202439         NaN  
1       NaN                 0.924844         NaN  
2       NaN                 0.822725         NaN  
3       NaN                 0.669156         NaN  
4       NaN                 0.468447         NaN  


In [14]:
# df = df.drop(columns=['Team1ID', 'Team2ID'])
print(df.head())

          MatchID  PredictionScore MatchYear  Team1ID  Team2ID Team1Name  \
0  2025_1101_1102         0.797561      2025     1101     1102       NaN   
1  2025_1101_1103         0.075156      2025     1101     1103       NaN   
2  2025_1101_1104         0.177275      2025     1101     1104       NaN   
3  2025_1101_1105         0.330844      2025     1101     1105       NaN   
4  2025_1101_1106         0.531553      2025     1101     1106       NaN   

  Team2Name  PredictionScoreSwitched WinningTeam  
0       NaN                 0.202439         NaN  
1       NaN                 0.924844         NaN  
2       NaN                 0.822725         NaN  
3       NaN                 0.669156         NaN  
4       NaN                 0.468447         NaN  


In [15]:
# Rename columns
df = df.rename(columns={
    'PredictionScore': 'P_Score_1',
    'PredictionScoreSwitched': 'P_Score_2'
})

# Drop the MatchYear column
df = df.drop(columns=['MatchYear'])

# Display the updated dataframe
print(df.head())

          MatchID  P_Score_1  Team1ID  Team2ID Team1Name Team2Name  P_Score_2  \
0  2025_1101_1102   0.797561     1101     1102       NaN       NaN   0.202439   
1  2025_1101_1103   0.075156     1101     1103       NaN       NaN   0.924844   
2  2025_1101_1104   0.177275     1101     1104       NaN       NaN   0.822725   
3  2025_1101_1105   0.330844     1101     1105       NaN       NaN   0.669156   
4  2025_1101_1106   0.531553     1101     1106       NaN       NaN   0.468447   

  WinningTeam  
0         NaN  
1         NaN  
2         NaN  
3         NaN  
4         NaN  


In [16]:
# Rearrange the columns
df = df[['WinningTeam', 'Team1Name', 'Team1ID', 'Team2Name', 'Team2ID','P_Score_1', 'P_Score_2', 'MatchID']]

# Display the updated dataframe
print(df.head())

  WinningTeam Team1Name  Team1ID Team2Name  Team2ID  P_Score_1  P_Score_2  \
0         NaN       NaN     1101       NaN     1102   0.797561   0.202439   
1         NaN       NaN     1101       NaN     1103   0.075156   0.924844   
2         NaN       NaN     1101       NaN     1104   0.177275   0.822725   
3         NaN       NaN     1101       NaN     1105   0.330844   0.669156   
4         NaN       NaN     1101       NaN     1106   0.531553   0.468447   

          MatchID  
0  2025_1101_1102  
1  2025_1101_1103  
2  2025_1101_1104  
3  2025_1101_1105  
4  2025_1101_1106  


In [17]:
temple_north_texas_matches = df[((df['Team1Name'] == 'Temple') & (df['Team2Name'] == 'North Texas')) | 
                                ((df['Team1Name'] == 'North Texas') & (df['Team2Name'] == 'Temple'))]
print(temple_north_texas_matches)

        WinningTeam    Team1Name  Team1ID Team2Name  Team2ID  P_Score_1  \
119236  North Texas  North Texas     3317    Temple     3396    0.64118   

        P_Score_2         MatchID  
119236    0.35882  2025_3317_3396  


In [18]:
washington_matches = df[(df['Team1Name'].str.contains('Washington', case=False, na=False)) | 
                        (df['Team2Name'].str.contains('Washington', case=False, na=False))]
print(washington_matches)

          WinningTeam      Team1Name  Team1ID       Team2Name  Team2ID  \
66143     Abilene Chr    Abilene Chr     3101    E Washington     3186   
66160     Abilene Chr    Abilene Chr     3101    G Washington     3203   
66395     Abilene Chr    Abilene Chr     3101      Washington     3449   
66396     Abilene Chr    Abilene Chr     3101   Washington St     3450   
66503       Air Force      Air Force     3102    E Washington     3186   
...               ...            ...      ...             ...      ...   
130967  Washington St  Washington St     3450       Stonehill     3476   
130968  Washington St  Washington St     3450  East Texas A&M     3477   
130969  Washington St  Washington St     3450        Le Moyne     3478   
130970  Washington St  Washington St     3450      Mercyhurst     3479   
130971  Washington St  Washington St     3450    West Georgia     3480   

        P_Score_1  P_Score_2         MatchID  
66143    0.809017   0.190983  2025_3101_3186  
66160    0.749785

In [19]:
washington_as_any_team = df[(df['Team1ID'] == 1449) | (df['Team2ID'] == 1449)]
print(washington_as_any_team)


      WinningTeam Team1Name  Team1ID Team2Name  Team2ID  P_Score_1  P_Score_2  \
331           NaN       NaN     1101       NaN     1449   0.487789   0.512211   
693           NaN       NaN     1102       NaN     1449   0.256253   0.743747   
1054          NaN       NaN     1103       NaN     1449   0.870380   0.129620   
1414          NaN       NaN     1104       NaN     1449   0.737756   0.262244   
1773          NaN       NaN     1105       NaN     1449   0.344822   0.655178   
...           ...       ...      ...       ...      ...        ...        ...   
65596         NaN       NaN     1449       NaN     1476   0.508596   0.491404   
65597         NaN       NaN     1449       NaN     1477   0.665995   0.334005   
65598         NaN       NaN     1449       NaN     1478   0.424487   0.575513   
65599         NaN       NaN     1449       NaN     1479   0.608448   0.391552   
65600         NaN       NaN     1449       NaN     1480   0.804949   0.195051   

              MatchID  
331

In [20]:
ohio_state_matches = df[(df['Team1Name'].str.contains('Ohio St', case=False, na=False)) | 
                        (df['Team2Name'].str.contains('Ohio St', case=False, na=False))]
print(ohio_state_matches)

       WinningTeam    Team1Name  Team1ID       Team2Name  Team2ID  P_Score_1  \
66279      Ohio St  Abilene Chr     3101         Ohio St     3326   0.130986   
66639      Ohio St    Air Force     3102         Ohio St     3326   0.173660   
66998      Ohio St        Akron     3103         Ohio St     3326   0.169155   
67356      Alabama      Alabama     3104         Ohio St     3326   0.574808   
67713      Ohio St  Alabama A&M     3105         Ohio St     3326   0.140841   
...            ...          ...      ...             ...      ...        ...   
120671     Ohio St      Ohio St     3326       Stonehill     3476   0.919658   
120672     Ohio St      Ohio St     3326  East Texas A&M     3477   0.962603   
120673     Ohio St      Ohio St     3326        Le Moyne     3478   0.955292   
120674     Ohio St      Ohio St     3326      Mercyhurst     3479   0.972392   
120675     Ohio St      Ohio St     3326    West Georgia     3480   0.972261   

        P_Score_2         MatchID  
662

In [21]:
ohio_st_not_winning = df[((df['Team1Name'] == 'Ohio St') | (df['Team2Name'] == 'Ohio St')) & (df['WinningTeam'] != 'Ohio St')]
print(ohio_st_not_winning)
teams_ohio_st_loses_to = ohio_st_not_winning['WinningTeam'].unique().tolist()
print(teams_ohio_st_loses_to)

           WinningTeam       Team1Name  Team1ID       Team2Name  Team2ID  \
67356          Alabama         Alabama     3104         Ohio St     3326   
73289           Baylor          Baylor     3124         Ohio St     3326   
83054   Col Charleston  Col Charleston     3158         Ohio St     3326   
84594      Connecticut     Connecticut     3163         Ohio St     3326   
89931             Duke            Duke     3181         Ohio St     3326   
93033      F Dickinson     F Dickinson     3192         Ohio St     3326   
94944       Florida St      Florida St     3199         Ohio St     3326   
96806     George Mason    George Mason     3206         Ohio St     3326   
98619     Grand Canyon    Grand Canyon     3213         Ohio St     3326   
99128          Harvard         Harvard     3217         Ohio St     3326   
105381       Kansas St       Kansas St     3243         Ohio St     3326   
109314             LSU             LSU     3261         Ohio St     3326   
112344      

In [22]:
df.to_csv('w_madness_pred.csv', index=False)